In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torch.utils.data as dt
import torchvision.transforms as transforms
import time
import os
import random
from PIL import Image
from dataset import AutoDataset
from networks import AutoEncoder
%matplotlib widget

In [ ]:
class FullyConnected(nn.Models):
    def __init__(self, name = 'Fully'):
        super(FullyConnected, self).__init__()
        self.fc1 = nn.Linear(128*2*2, 32)
        self.fc2 = nn.Linear(32, 14)
    
    def forward(self, x):
        x = x.view(-1, 128*2*2)

        # use relu as activation function
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x

In [ ]:
auto = AutoEncoder()
model = torch.load('Model\epoch30_batch64_lr0.0001.pth')
auto.load_state_dict(model)

In [ ]:
def training(model, bs = 27, ne = 1, lr = 0.001, hilbert = True):
    '''
    train the data
    '''
    criterion = nn.MultiLabelSoftMarginLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # load in data and create accuracy arrays
    train_loader, val_loader, test_loader = load_data(bs, hilbert)
    train_loss, train_acc, val_acc, iters = [], [], [], []

    # Training
    start_time = time.time()
    i = 0
    print ("Training Started...")
    for epoch in range(ne):
        lo = 0
        j = 0
        for feature, label in iter(train_loader):

            # Run on GPU if possible
            if torch.cuda.is_available():
                features = feature[0].squeeze(1).cuda()
                labels = feature[1].squeeze(1).cuda()
            else:
                features = feature[0].squeeze(1)
                labels = feature[1].squeeze(1)
            if len(features) == bs:
                
                features = auto(features).cuda()

                output = model(features)           # forward pass

                loss = criterion(output, labels) # compute loss
                loss.backward()                  # backward pass
                optimizer.step()                 # update parameter
                optimizer.zero_grad()            # clean up
                lo += loss
                j += 1
                #print(output)
                #print(labels)
        iters.append(i)
        i+=1
        train_loss.append(float(lo)/bs/j)           # compute loss
        train_acc.append(get_accuracy(model, train_loader)[0]) # compute train_acc
        val_acc.append(get_accuracy(model, val_loader)[0])   # compute val_acc
        print("Epoch: " + str(epoch) + ', train acc: ' + str(train_acc[-1]) + ', train loss: ' + str(float(train_loss[-1])) + ', valid acc: ' + str(val_acc[-1]))
        if hilbert == True:
            n = "Hilbert"
        else:
            n = "Spectrogram"
        model_path = "/Users/sarinaxi/Desktop/Lingling-Bot/Data/" + n + "/autoCNN/models/model_customlabel_{0}_bs{1}_lr{2}_epoch{3}".format(model.name, bs, lr, ne)
        torch.save(model.state_dict(), model_path)

    print('Finished Training')
    end_time = time.time()
    elapsed = end_time - start_time
    print("Total time elapsed: " + str(elapsed/60/60) + " hours.")
    print("Final Training Accuracy: {}".format(train_acc[-1]))
    print("Final Validation Accuracy: {}".format(val_acc[-1]))
    return iters, train_loss, train_acc, val_acc, model.name, bs, lr, ne, test_loader